# Guide Context Generator

Traverses a project folder recursively, reads every file in chunks,
uses a local LLM to produce a **2-line summary + keywords** per chunk,
then collapses all chunk summaries into a **final 2-line file summary**.
The result is a structured Markdown guide suitable for ingestion by `GuideModel`.

## Pipeline

```
Project Folder
     │
     ▼
Recursive File Walker  (respects skip-dir rules)
     │
     ▼
Chunk Reader           (splits each file into N-char chunks)
     │
     ▼
LLM Chunk Summarizer   (parallel slots: N chunks at once)
     │
     ▼
LLM File Summarizer    (2-line summary from all chunk summaries)
     │
     ▼
Guide Builder          (assembles Markdown: path → final summary)
     │
     ▼
guides/<project>.md    (ready for GuideModel.load_guide())
```

## 1. Install Dependencies

In [1]:
!pip install llama-cpp-python python-dotenv huggingface-hub pathspec


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Configuration

In [2]:
import os

# ── Project to scan ────────────────────────────────────────────────────────────
PROJECT_PATH   = "/home/codx-junior-projects/codx-junior"
PLUGIN_PATH    = os.path.join(PROJECT_PATH, "plugins/guide-model")
GUIDES_DIR     = os.path.join(PROJECT_PATH, "guides")
GUIDE_OUT_FILE = os.path.join(GUIDES_DIR, "codx-junior.md")

GUIDE_TARGET_PATH = "/home/codx-junior-projects/codx-junior/api/codx/junior/agents"

# ── Model settings ─────────────────────────────────────────────────────────────
CODX_JUNIOR_MODELS_PATH = os.environ.get(
    "CODX_JUNIOR_MODELS_PATH",
    os.path.join(PROJECT_PATH, "models"),
)
MODEL_ID        = "unsloth/Qwen3.5-0.8B-GGUF"
MODEL_NAME      = "Qwen3.5-0.8B-Q8_0.gguf"
RESERVED_TOKENS = 512     # tokens kept for the answer

# ── Parallel slots ─────────────────────────────────────────────────────────────
# N_PARALLEL: number of decode slots allocated in the llama.cpp context.
# Each slot gets N_CTX / N_PARALLEL tokens of working context.
# ~600 tokens needed per slot (400 chunk + 120 answer + 80 scaffold)
# → 2048 per slot is comfortable.
N_PARALLEL      = 4
N_CTX           = N_PARALLEL * 2048   # = 8192 total

# ── Walker settings ────────────────────────────────────────────────────────────
SKIP_DIRS = {
    ".git", "__pycache__", ".venv", "venv", "node_modules",
    ".mypy_cache", ".pytest_cache", ".ruff_cache",
    "dist", "build", ".next", ".nuxt",
    "models",
    "guides",
    "notebooks",
}

# ── Chunking settings ──────────────────────────────────────────────────────────
# ~1 500 chars ≈ ~400 tokens — leaves plenty of room for the prompt scaffold.
CHUNK_SIZE_CHARS    = 1_500
MAX_CHUNKS_PER_FILE = 20
MAX_FILES           = 200
MAX_FILE_BYTES      = 500_000   # 500 KB

os.makedirs(GUIDES_DIR, exist_ok=True)

print(f"Project path   : {PROJECT_PATH}")
print(f"Guide output   : {GUIDE_OUT_FILE}")
print(f"Models path    : {CODX_JUNIOR_MODELS_PATH}")
print(f"Chunk size     : {CHUNK_SIZE_CHARS} chars")
print(f"Max chunks     : {MAX_CHUNKS_PER_FILE} / file")
print(f"N_PARALLEL     : {N_PARALLEL} slots")
print(f"N_CTX          : {N_CTX} ({N_CTX // N_PARALLEL} per slot)")

Project path   : /home/codx-junior-projects/codx-junior
Guide output   : /home/codx-junior-projects/codx-junior/guides/codx-junior.md
Models path    : /home/codx-junior-projects/codx-junior/models
Chunk size     : 1500 chars
Max chunks     : 20 / file
N_PARALLEL     : 4 slots
N_CTX          : 8192 (2048 per slot)


## 3. Load the LLM (Summarizer)

Key change: `n_parallel=N_PARALLEL` allocates multiple decode slots inside
llama.cpp so that concurrent requests are batched efficiently.

In [3]:
import time
import multiprocessing
from llama_cpp import Llama

MODEL_PATH = os.path.join(CODX_JUNIOR_MODELS_PATH, MODEL_NAME)

print(f"Loading model  : {MODEL_PATH}")
print(f"Parallel slots : {N_PARALLEL}")
print(f"Total n_ctx    : {N_CTX}  ({N_CTX // N_PARALLEL} per slot)")
t0 = time.time()

_llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_parallel=N_PARALLEL,          # ← allocates parallel decode slots
    n_threads=max(1, multiprocessing.cpu_count() // 2),
    verbose=False,
)

print(f"Model loaded in {time.time()-t0:.2f}s ✓")
print(f"n_ctx={N_CTX}, n_parallel={N_PARALLEL}")

Loading model  : /home/codx-junior-projects/codx-junior/models/Qwen3.5-0.8B-Q8_0.gguf
Parallel slots : 4
Total n_ctx    : 8192  (2048 per slot)


llama_context: n_ctx_seq (8192) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


Model loaded in 0.72s ✓
n_ctx=8192, n_parallel=4


## 4. Recursive File Walker

In [4]:
from pathlib import Path
from typing import Iterator


def is_binary_file(path: Path, sample_bytes: int = 8192) -> bool:
    """Heuristic: a file is binary if its first *sample_bytes* contain a NUL byte."""
    try:
        with open(path, "rb") as fh:
            chunk = fh.read(sample_bytes)
        return b"\x00" in chunk
    except Exception:
        return True


def iter_project_files(
    root: str,
    skip_dirs: set,
    max_files: int = MAX_FILES,
) -> Iterator[Path]:
    """
    Yields Path objects for every **text** file under *root*,
    skipping directories in *skip_dirs* and binary files.
    Files are yielded in deterministic (sorted) order.
    """
    root_path = Path(root)
    count = 0

    for dirpath, dirnames, filenames in os.walk(root_path, topdown=True):
        dirnames[:] = sorted(
            d for d in dirnames
            if d not in skip_dirs and not d.startswith(".")
        )

        for filename in sorted(filenames):
            if count >= max_files:
                return
            filepath = Path(dirpath) / filename
            try:
                if filepath.stat().st_size > MAX_FILE_BYTES:
                    continue
            except OSError:
                continue
            if is_binary_file(filepath):
                continue
            yield filepath
            count += 1


# Quick sanity check
sample_files = list(iter_project_files(PROJECT_PATH, SKIP_DIRS, max_files=10))
print(f"Sample files found ({len(sample_files)}):")
for f in sample_files:
    print(f"  {f.relative_to(PROJECT_PATH)}")

Sample files found (10):
  .dockerignore
  .gitignore
  LICENSE.md
  README.md
  build-docker.sh
  codx-junior
  codx-junior-python.sh
  codx-junior.code-workspace
  codx-junior.installed
  config.json


## 5. Chunk Reader

In [5]:
from typing import List


def read_file_chunks(
    path: Path,
    chunk_size: int = CHUNK_SIZE_CHARS,
    max_chunks: int = MAX_CHUNKS_PER_FILE,
) -> List[str]:
    """
    Reads *path* as UTF-8 text and splits it into chunks of *chunk_size*
    characters.  At most *max_chunks* chunks are returned.

    Chunk boundaries are nudged to the nearest newline so that lines are
    never split mid-way (best-effort).
    """
    try:
        text = path.read_text(encoding="utf-8", errors="replace")
    except Exception as exc:
        print(f"  [skip] Cannot read {path}: {exc}")
        return []

    if not text.strip():
        return []

    chunks: List[str] = []
    start = 0
    total = len(text)

    while start < total and len(chunks) < max_chunks:
        end = min(start + chunk_size, total)
        if end < total:
            nl_pos = text.find("\n", end, end + 200)
            if nl_pos != -1:
                end = nl_pos + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end

    return chunks


# Smoke test
if sample_files:
    test_chunks = read_file_chunks(sample_files[0])
    print(f"File  : {sample_files[0].name}")
    print(f"Chunks: {len(test_chunks)}")
    if test_chunks:
        print(f"First chunk preview: {test_chunks[0][:120]!r}")

print("Chunk reader defined ✓")

File  : .dockerignore
Chunks: 1
First chunk preview: '**/.git\n.venv\nnode_modules\n__pycache__\n*egg-*\nclient/.vite\n**/global_settings.json\n**/.codx/db\nfilebrowser.db\n.codx\n.pyt'
Chunk reader defined ✓


## 6. LLM Summarizer (parallel slots)

Two prompt templates:
- **chunk prompt** — given a raw code/text chunk, return a 2-line summary + keywords
- **file prompt**  — given all chunk summaries for one file, collapse to a final 2-line summary

### Parallelism strategy

- `llama_cpp` compiled with `n_parallel > 1` batches requests across slots internally.
- A `threading.Semaphore(N_PARALLEL)` caps concurrent Python-side calls to exactly
  the number of slots, preventing slot starvation and OOM.
- The GIL is released inside the C++ `llama_decode` call, so `ThreadPoolExecutor`
  gives real parallelism without the overhead of `multiprocessing`.

In [6]:
from typing import List, TypedDict
import re
import json
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed


# ── Semaphore — caps concurrent LLM calls to N_PARALLEL ──────────────────────
# Must be recreated here (after N_PARALLEL is defined) so re-running the cell
# picks up any config changes.
_llm_semaphore = threading.Semaphore(N_PARALLEL)


# ── Output schema ─────────────────────────────────────────────────────────────

class ChunkSummary:
    def __init__(self, description: str, keywords: List[str]):
        self.description = description
        self.keywords    = keywords

    def __str__(self) -> str:
        kw = ", ".join(self.keywords)
        return f"{self.description}. Keywords: {kw}"

    def __getitem__(self, key: str):
        return getattr(self, key)

    def get(self, key: str, default=None):
        return getattr(self, key, default)


class FileSummary:
    def __init__(self, path: str, description: str, keywords: List[str]):
        self.path        = path
        self.description = description
        self.keywords    = keywords

    def __str__(self) -> str:
        kw = ", ".join(self.keywords)
        return f"{self.description}. Keywords: {kw}"

    def __getitem__(self, key: str):
        return getattr(self, key)

    def get(self, key: str, default=None):
        return getattr(self, key, default)


# ── Prompt templates ──────────────────────────────────────────────────────────

_CHUNK_PROMPT_TMPL = """<|im_start|>system
You are a concise code documentation assistant.
Reply with a single JSON object and nothing else.
Schema: {{"description": "<max 20 words>", "keywords": ["kw1", "kw2", ...]}}<|im_end|>
<|im_start|>user
File: {file_path} (chunk {chunk_index}/{total_chunks})

{chunk_text}

Write a JSON object:
- "description": what this chunk does (max 20 words).
- "keywords": array of strings (max 10).<|im_end|>
<|im_start|>assistant
"""

_FILE_PROMPT_TMPL = """<|im_start|>system
You are a concise code documentation assistant.
Reply with a single JSON object and nothing else.
Schema: {{"description": "<max 20 words>", "keywords": ["kw1", "kw2", ...]}}<|im_end|>
<|im_start|>user
Chunk summaries for file: {file_path}

{chunk_summaries}

Write a JSON object:
- "description": what the whole file does (max 20 words).
- "keywords": array of strings (max 10).<|im_end|>
<|im_start|>assistant
"""


# ── Helpers ───────────────────────────────────────────────────────────────────

def _strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks emitted by reasoning models."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.strip()


def _extract_json(text: str) -> dict:
    """
    Extracts the first JSON object found in *text*.
    Falls back to a default structure if parsing fails.
    """
    text = _strip_thinking(text)
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"description": text[:120].strip(), "keywords": []}


def _normalize_summary(raw: dict) -> ChunkSummary:
    """
    Ensures the parsed dict conforms to ChunkSummary schema.
    Coerces types and enforces limits.
    """
    description = str(raw.get("description", "(no description)")).strip()
    keywords_raw = raw.get("keywords", [])

    if isinstance(keywords_raw, str):
        keywords = [k.strip() for k in keywords_raw.split(",") if k.strip()]
    elif isinstance(keywords_raw, list):
        keywords = [str(k).strip() for k in keywords_raw if k]
    else:
        keywords = []

    return ChunkSummary(
        description=description[:200],
        keywords=keywords[:10],
    )


# ── Core LLM call — semaphore-guarded ─────────────────────────────────────────

def _call_llm(prompt: str, max_tokens: int = 120) -> str:
    """
    Sends *prompt* to the loaded LLM and returns the stripped response text.

    The semaphore ensures at most N_PARALLEL calls run concurrently,
    matching the number of decode slots allocated at model load time.
    """
    with _llm_semaphore:
        result = _llm(
            prompt,
            max_tokens=max_tokens,
            temperature=0.1,
            top_p=0.9,
            stop=["\n\n\n", "File:", "Chunk ", "<|im_start|>"],
            echo=False,
        )
    raw = result["choices"][0]["text"].strip()
    return _strip_thinking(raw)


# ── Public API ────────────────────────────────────────────────────────────────

def summarize_chunk(
    file_path: str,
    chunk_text: str,
    chunk_index: int,
    total_chunks: int,
    chunk_size_chars: int = 1500,
) -> ChunkSummary:
    """
    Returns a ChunkSummary (description + keywords list) for one file chunk.
    Falls back to a safe default if the LLM fails or returns malformed JSON.
    """
    prompt = _CHUNK_PROMPT_TMPL.format(
        file_path=file_path,
        chunk_index=chunk_index + 1,
        total_chunks=total_chunks,
        chunk_text=chunk_text[:chunk_size_chars],
    )
    try:
        raw_text = _call_llm(prompt, max_tokens=120)
        return _normalize_summary(_extract_json(raw_text))
    except Exception as exc:
        return ChunkSummary(
            description=f"(summarization failed: {exc})",
            keywords=[],
        )


def summarize_file(
    file_path: str,
    chunk_summaries: List[ChunkSummary],
) -> FileSummary:
    """
    Collapses all per-chunk ChunkSummary objects into a single FileSummary.
    If there is only one chunk, its summary is returned directly.
    """
    if not chunk_summaries:
        return FileSummary(
            path=file_path,
            description="(empty or unreadable file)",
            keywords=[],
        )

    if len(chunk_summaries) == 1:
        s = chunk_summaries[0]
        return FileSummary(
            path=file_path,
            description=s["description"],
            keywords=s["keywords"],
        )

    numbered = "\n".join(
        f"[{i+1}] {s['description']} | keywords: {', '.join(s['keywords'])}"
        for i, s in enumerate(chunk_summaries)
    )
    prompt = _FILE_PROMPT_TMPL.format(
        file_path=file_path,
        chunk_summaries=numbered,
    )
    try:
        raw_text = _call_llm(prompt, max_tokens=120)
        normalized = _normalize_summary(_extract_json(raw_text))
        return FileSummary(
            path=file_path,
            description=normalized["description"],
            keywords=normalized["keywords"],
        )
    except Exception as exc:
        merged_desc = " | ".join(s["description"] for s in chunk_summaries)
        merged_kw   = list({kw for s in chunk_summaries for kw in s["keywords"]})
        return FileSummary(
            path=file_path,
            description=merged_desc[:200],
            keywords=merged_kw[:10],
        )


# ── Parallel file summarizer ──────────────────────────────────────────────────

def summarize_file_parallel(fp: Path, root_path: Path) -> tuple:
    """
    Reads, chunks, and summarizes a single file using parallel slot calls.
    All chunks of the file are dispatched concurrently into the slot pool.
    Returns (rel_path_str, FileSummary).

    Designed to run inside a ThreadPoolExecutor worker.
    """
    rel    = str(fp.relative_to(root_path))
    chunks = read_file_chunks(fp)

    if not chunks:
        return rel, FileSummary(
            path=rel,
            description="(empty or unreadable file)",
            keywords=[],
        )

    # Fan out all chunks of this file concurrently.
    # Workers are capped to N_PARALLEL — no point exceeding slot count.
    chunk_sums: List[ChunkSummary] = [None] * len(chunks)

    def _do_chunk(args):
        idx, chunk = args
        return idx, summarize_chunk(rel, chunk, idx, len(chunks))

    with ThreadPoolExecutor(max_workers=min(len(chunks), N_PARALLEL)) as pool:
        futures = {pool.submit(_do_chunk, (i, c)): i for i, c in enumerate(chunks)}
        for future in as_completed(futures):
            idx, summary = future.result()
            chunk_sums[idx] = summary

    return rel, summarize_file(rel, chunk_sums)


print(f"LLM summarizer (parallel, {N_PARALLEL} slots) defined ✓")

LLM summarizer (parallel, 4 slots) defined ✓


## 7. Smoke-Test the Summarizer on One File

In [7]:
if sample_files:
    test_fp  = sample_files[0]
    test_rel = str(test_fp.relative_to(PROJECT_PATH))

    print(f"Testing parallel summarizer on: {test_rel}")
    t0 = time.time()
    rel_out, file_sum = summarize_file_parallel(test_fp, Path(PROJECT_PATH))
    elapsed = time.time() - t0

    print(f"Completed in {elapsed:.2f}s")
    print(f"File summary : {str(file_sum)}")

Testing parallel summarizer on: .dockerignore
Completed in 8.98s
File summary : <think>
The user wants me to write a JSON object describing a .dockerignore file chunk. Let me analyze what this file co. Keywords: 


## 8. Directory Tree Builder

In [8]:
from pathlib import Path


def build_directory_tree(
    root: str,
    skip_dirs: set,
    max_depth: int = 3,
    max_children: int = 20,
) -> str:
    """
    Returns a compact ASCII directory tree string limited to *max_depth* levels.
    """
    lines: List[str] = []
    root_path = Path(root)

    def _walk(path: Path, prefix: str, depth: int) -> None:
        if depth > max_depth:
            return
        try:
            children = sorted(path.iterdir())
        except PermissionError:
            return

        children = [
            c for c in children
            if not c.name.startswith(".") and c.name not in skip_dirs
        ]
        dirs    = [c for c in children if c.is_dir()]
        files   = [c for c in children if c.is_file()]
        visible = (dirs + files)[:max_children]
        hidden  = len(dirs) + len(files) - len(visible)

        for i, child in enumerate(visible):
            is_last   = (i == len(visible) - 1) and hidden == 0
            connector = "└── " if is_last else "├── "
            suffix    = "/" if child.is_dir() else ""
            lines.append(f"{prefix}{connector}{child.name}{suffix}")
            if child.is_dir():
                extension = "    " if is_last else "│   "
                _walk(child, prefix + extension, depth + 1)

        if hidden > 0:
            lines.append(f"{prefix}└── … ({hidden} more)")

    lines.append(f"{root_path.name}/")
    _walk(root_path, "", 1)
    return "\n".join(lines)


tree = build_directory_tree(GUIDE_TARGET_PATH, SKIP_DIRS, max_depth=3)
print(tree[:1200])
print(f"\n… total lines: {len(tree.splitlines())}")

agents/
├── base_agent.py
├── devops_agent.py
└── git_issues_agent.py

… total lines: 4


## 9. Guide Builder

Assembles the final Markdown guide from `{file_path: final_2line_summary}` mapping.

In [ ]:
from typing import Dict


def build_guide(
    project_name: str,
    directory_tree: str,
    file_summaries: Dict[str, str],
) -> str:
    """
    Assembles a Markdown guide from a mapping of
    ``{relative_file_path: two_line_summary}``.

    Structure:
    1. Header + directory tree
    2. One entry per file::

         ### `path/to/file.py`
         <description line>
         **Keywords:** kw1, kw2, …
    """
    sections: List[str] = []

    # ── 1. Header ─────────────────────────────────────────────────────────────
    sections.append(f"""# {project_name} — Project Guide

> Auto-generated by `guide_context_generator.ipynb` (AI-powered summaries).
> Do **not** edit manually — re-run the notebook to refresh.

## Directory Structure

```
{directory_tree}
```
""")

    # ── 2. File Summaries ──────────────────────────────────────────────────────
    sections.append("## File Summaries\n")

    for rel_path in sorted(file_summaries.keys()):
        summary = file_summaries[rel_path].strip()
        lines   = summary.splitlines()

        description = lines[0].strip() if len(lines) > 0 else "(no description)"
        keywords    = lines[1].strip() if len(lines) > 1 else ""

        entry = [f"### `{rel_path}`", f"", description]
        if keywords:
            kw_clean = keywords.lstrip("Keywords: ").lstrip("keywords: ").strip()
            entry.append(f"**Keywords:** {kw_clean}")
        entry.append("")   # blank line between entries
        sections.append("\n".join(entry))

    return "\n\n---\n\n".join(sections[:2]) + "\n\n" + "\n".join(sections[2:])


print("Guide builder defined ✓")

Guide builder defined ✓


: 

## 10. Run the Full Pipeline

### Parallelism layout

```
ThreadPoolExecutor (FILE_WORKERS = N_PARALLEL)
  └─ summarize_file_parallel(file_A)        ← worker thread 0
       └─ ThreadPoolExecutor (N_PARALLEL workers)
            ├─ summarize_chunk(chunk_0)      ─┐
            ├─ summarize_chunk(chunk_1)       │ all guarded by
            └─ summarize_chunk(chunk_2)      ─┘ _llm_semaphore
  └─ summarize_file_parallel(file_B)        ← worker thread 1
       └─ …
```

The semaphore ensures total concurrent `_llm()` calls ≤ `N_PARALLEL` at all times.

In [ ]:
import time
from pathlib import Path
from typing import Dict
from concurrent.futures import ThreadPoolExecutor, as_completed

root_path    = Path(GUIDE_TARGET_PATH)
project_name = root_path.name

print(f"Scanning    : {PROJECT_PATH}")
print(f"Skip dirs   : {sorted(SKIP_DIRS)}")
print(f"Parallelism : {N_PARALLEL} slots")
print()

t_pipeline_start = time.time()

# ── Step 1: Walk the file tree ─────────────────────────────────────────────
t1 = time.time()
all_files = list(iter_project_files(PROJECT_PATH, SKIP_DIRS, max_files=MAX_FILES))
print(f"Step 1 — Files found: {len(all_files)}  ({time.time()-t1:.2f}s)")

# ── Step 2: Summarize files in parallel ───────────────────────────────────
# FILE_WORKERS outer threads dispatch files concurrently.
# Each file internally fans out its chunks across the slot pool.
# The semaphore keeps total active LLM calls ≤ N_PARALLEL.
t2 = time.time()
file_summaries: Dict[str, str] = {}
total_chunks_processed = 0
completed_count        = 0
FILE_WORKERS           = N_PARALLEL

with ThreadPoolExecutor(max_workers=FILE_WORKERS) as file_pool:
    future_to_fp = {
        file_pool.submit(summarize_file_parallel, fp, root_path): fp
        for fp in all_files
    }

    for future in as_completed(future_to_fp):
        fp = future_to_fp[future]
        completed_count += 1
        try:
            rel, summary = future.result()
            file_summaries[rel] = str(summary)
            n_chunks = len(read_file_chunks(fp))
            total_chunks_processed += n_chunks
        except Exception as exc:
            rel = str(fp.relative_to(root_path))
            file_summaries[rel] = f"(pipeline error: {exc})"
            print(f"  [ERROR] {rel}: {exc}")

        if completed_count % 10 == 0 or completed_count == len(all_files):
            elapsed = time.time() - t2
            rate    = completed_count / elapsed if elapsed > 0 else 0
            print(
                f"  [{completed_count:>3}/{len(all_files)}] "
                f"{str(fp.relative_to(root_path)):<55} "
                f"{rate:.1f} files/s"
            )

print(
    f"Step 2 — Files summarised: {len(file_summaries)}  "
    f"total chunks: {total_chunks_processed}  "
    f"({time.time()-t2:.2f}s)"
)

# ── Step 3: Build directory tree ──────────────────────────────────────────
t3 = time.time()
dir_tree = build_directory_tree(PROJECT_PATH, SKIP_DIRS, max_depth=3)
print(f"Step 3 — Directory tree built  ({time.time()-t3:.2f}s)")

# ── Step 4: Assemble the guide ────────────────────────────────────────────
t4 = time.time()
guide_md = build_guide(project_name, dir_tree, file_summaries)
print(
    f"Step 4 — Guide assembled: {len(guide_md):,} chars, "
    f"{len(guide_md.splitlines())} lines  ({time.time()-t4:.2f}s)"
)

# ── Step 5: Write to disk ─────────────────────────────────────────────────
t5 = time.time()
with open(GUIDE_OUT_FILE, "w", encoding="utf-8") as fh:
    fh.write(guide_md)
size_kb = os.path.getsize(GUIDE_OUT_FILE) / 1024
print(
    f"Step 5 — Written to: {GUIDE_OUT_FILE}  "
    f"({size_kb:.1f} KB)  ({time.time()-t5:.2f}s)"
)

print(f"\nTotal pipeline time: {time.time()-t_pipeline_start:.2f}s")

## 11. Preview the Generated Guide

In [ ]:
PREVIEW_CHARS = 3000

print(f"{'='*60}")
print(f"Guide preview (first {PREVIEW_CHARS} chars)")
print(f"{'='*60}")
print(guide_md[:PREVIEW_CHARS])
print(f"\n… ({len(guide_md):,} total characters)")

## 12. Token Count & Budget Check

Verify the guide fits inside the target `GuideModel` context window.

In [ ]:
GUIDE_N_CTX     = 262144
GUIDE_RESERVED  = 512
SCAFFOLD_TOKENS = 32

guide_tokens     = len(_llm.tokenize(guide_md.encode("utf-8"), add_bos=False))
max_guide_tokens = GUIDE_N_CTX - GUIDE_RESERVED - SCAFFOLD_TOKENS
headroom         = max_guide_tokens - guide_tokens
fits             = guide_tokens <= max_guide_tokens

print(f"""
Token budget analysis
─────────────────────────────────────────────────────
  Guide tokens      : {guide_tokens:>10,}
  Max guide tokens  : {max_guide_tokens:>10,}  (n_ctx={GUIDE_N_CTX} - reserved={GUIDE_RESERVED} - scaffold={SCAFFOLD_TOKENS})
  Headroom          : {headroom:>10,}  {'✓ fits' if fits else '✗ TOO LARGE'}
─────────────────────────────────────────────────────
""")

if not fits:
    overage = -headroom
    pct = overage / max_guide_tokens * 100
    print(f"⚠  Guide is {overage:,} tokens ({pct:.1f}%) over budget.")
    print(f"   Options:")
    print(f"     • Increase GUIDE_N_CTX (currently {GUIDE_N_CTX})")
    print(f"     • Add more directories to SKIP_DIRS")
    print(f"     • Lower MAX_FILES (currently {MAX_FILES})")
    print(f"     • Lower MAX_CHUNKS_PER_FILE (currently {MAX_CHUNKS_PER_FILE})")
else:
    print("✓ Guide fits within the context budget.")

## 13. Smoke-Test with GuideModel

Load the generated guide into a `GuideModel` instance and run a quick Q&A.

In [ ]:
import sys, pickle, multiprocessing
if PLUGIN_PATH not in sys.path:
    sys.path.insert(0, PLUGIN_PATH)


# ── Inline GuideModel ─────────────────────────────────────────────────────────

class GuideModelError(Exception):
    pass

class GuideTooLargeError(GuideModelError):
    def __init__(self, guide_tokens, max_guide_tokens, n_ctx, reserved_tokens, scaffold_tokens):
        self.guide_tokens     = guide_tokens
        self.max_guide_tokens = max_guide_tokens
        super().__init__(
            f"Guide is too large: {guide_tokens} tokens (limit: {max_guide_tokens})."
        )

class NoGuideLoadedError(GuideModelError):
    def __init__(self):
        super().__init__("No guide loaded. Call load_guide() first.")


class GuideModel:
    _PROMPT_SCAFFOLD_TOKENS = 32

    def __init__(self, model_path, guide_id, n_ctx=4096, reserved_tokens=512):
        self.model_path      = model_path
        self.guide_id        = guide_id
        self.n_ctx           = n_ctx
        self.reserved_tokens = reserved_tokens
        model_dir            = os.path.dirname(model_path)
        model_basename       = os.path.basename(model_path)
        self.state_path      = os.path.join(model_dir, f"{model_basename}.{guide_id}.guide.state")
        self.guide_text      = ""
        self.guide_token_count = 0
        self.threads         = max(1, multiprocessing.cpu_count() // 2)
        self.llm             = Llama(
            model_path=model_path, n_ctx=n_ctx,
            n_threads=self.threads, verbose=False,
        )

    @property
    def max_guide_tokens(self):
        return self.n_ctx - self.reserved_tokens - self._PROMPT_SCAFFOLD_TOKENS

    def count_tokens(self, text):
        return len(self.llm.tokenize(text.encode("utf-8"), add_bos=False))

    def validate_guide_size(self, guide_text):
        count = self.count_tokens(guide_text)
        if count > self.max_guide_tokens:
            raise GuideTooLargeError(
                count, self.max_guide_tokens, self.n_ctx,
                self.reserved_tokens, self._PROMPT_SCAFFOLD_TOKENS,
            )
        return count

    def _save_state(self):
        state = self.llm.save_state()
        with open(self.state_path, "wb") as fh:
            pickle.dump(state, fh)
        print(f"[GuideModel:{self.guide_id}] State saved → {self.state_path}")

    def _restore_state(self):
        try:
            with open(self.state_path, "rb") as fh:
                state = pickle.load(fh)
            self.llm.load_state(state)
            print(f"[GuideModel:{self.guide_id}] State restored ← {self.state_path}")
            return True
        except Exception as exc:
            print(f"[GuideModel:{self.guide_id}] State restore failed: {exc}")
            return False

    def load_guide(self, guide_text, force_refresh=False):
        stripped    = guide_text.strip()
        token_count = self.validate_guide_size(stripped)
        self.guide_text        = stripped
        self.guide_token_count = token_count
        print(f"[GuideModel:{self.guide_id}] guide: {token_count}/{self.max_guide_tokens} tokens")
        if os.path.exists(self.state_path) and not force_refresh:
            if self._restore_state():
                return
        print(f"[GuideModel:{self.guide_id}] Processing guide ({token_count} tokens) …")
        self.llm(self.guide_text, max_tokens=1)
        self._save_state()

    def ask(self, question):
        if not self.guide_text:
            raise NoGuideLoadedError()
        full_prompt = f"{self.guide_text}\n\nQuestion: {question}\nAnswer:"
        output = self.llm(
            full_prompt,
            max_tokens=self.reserved_tokens - self._PROMPT_SCAFFOLD_TOKENS,
            stop=["Question:", "\n\n"],
            echo=False,
        )
        return output["choices"][0]["text"].strip()


print("GuideModel (inline) defined ✓")

In [ ]:
print("Initialising GuideModel with generated guide …")
t0 = time.time()

gm = GuideModel(
    model_path=MODEL_PATH,
    guide_id="codx-junior-generated",
    n_ctx=GUIDE_N_CTX,
    reserved_tokens=GUIDE_RESERVED,
)

try:
    gm.load_guide(guide_md)
    elapsed = time.time() - t0
    print(f"Guide loaded in {elapsed:.2f}s ✓")

    smoke_questions = [
        "What are the main Python modules in this project?",
        "Where is the FastAPI application entry point?",
        "Which files handle authentication?",
        "Where are environment variables configured?",
    ]

    print("\n" + "="*60)
    print("Smoke Q&A on generated guide")
    print("="*60)

    for q in smoke_questions:
        t1 = time.time()
        answer = gm.ask(q)
        elapsed = time.time() - t1
        print(f"\nQ: {q}")
        print(f"A: {answer}")
        print(f"   ({elapsed:.2f}s)")

except GuideTooLargeError as e:
    print(f"\n⚠  Guide too large: {e}")
    print(f"   • Lower MAX_FILES (current: {MAX_FILES})")
    print(f"   • Lower MAX_CHUNKS_PER_FILE (current: {MAX_CHUNKS_PER_FILE})")
    print(f"   • Add more paths to SKIP_DIRS")

## 14. Summary

In [ ]:
print("=" * 60)
print("Guide Context Generator — Summary")
print("=" * 60)
print(f"  Project            : {project_name}")
print(f"  Files scanned      : {len(all_files)}")
print(f"  Files summarised   : {len(file_summaries)}")
print(f"  Total chunks       : {total_chunks_processed}")
print(f"  Guide size         : {len(guide_md):,} chars  ({size_kb:.1f} KB)")
print(f"  Guide tokens       : {guide_tokens:,}")
print(f"  Output file        : {GUIDE_OUT_FILE}")
print(f"  N_PARALLEL slots   : {N_PARALLEL}")
print(f"  Token budget       : {'✓ fits' if fits else '✗ over budget'}")
print("=" * 60)
print("Pipeline complete ✓")